# 03 — Preguntas analíticas (Spark SQL)

**Fase 6.** Responde las 5 preguntas sobre el modelo dimensional `mobility_dw` (datos en `/movilidad/modeled/`).

| Preguntas | Destino | Visualización |
|-----------|---------|---------------|
| P1, P2, P3 | pandas DataFrame | Jupyter (mín. 2 gráficos) |
| P4, P5 | `/movilidad/analytics/` + Hive | SuperSet |

In [ ]:
# Dependencias: matplotlib, seaborn, pandas (instalar en la VM si faltan)
import sys

for pkg in ("matplotlib", "seaborn", "pandas"):
    try:
        __import__(pkg)
    except ImportError:
        raise ImportError(
            f"Falta {pkg}. En la VM: python3 -m pip install --user matplotlib seaborn pandas"
        )
print(f"Dependencias OK — {sys.executable}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("movilidad-preguntas")
    .enableHiveSupport()
    .getOrCreate()
)

spark.sql("USE mobility_dw")
MODELED = "hdfs:///movilidad/modeled"
ANALYTICS = "hdfs:///movilidad/analytics"

TABLAS_MODELED = [
    "dim_ubicacion", "dim_tiempo", "dim_detector", "dim_combustible",
    "dim_tipo_vehiculo", "dim_anio_parque", "dim_sensor",
    "fact_parque", "fact_conteo", "fact_velocidad",
]

for tabla in TABLAS_MODELED:
    spark.read.parquet(f"{MODELED}/{tabla}").createOrReplaceTempView(tabla)

spark.sparkContext.setLogLevel("WARN")
sns.set_theme(style="whitegrid")
print(f"Modelo dimensional listo — {len(TABLAS_MODELED)} tablas")

## P1 — ¿El crecimiento del parque se refleja en más circulación?

> ¿El crecimiento del parque automotor entre 2019 y 2024 se refleja en un aumento proporcional del volumen de circulación en las avenidas monitoreadas?

In [ ]:
p1_parque = spark.sql("""
SELECT a.anio_referencia, SUM(p.cantidad) AS total_parque
FROM fact_parque p
JOIN dim_anio_parque a ON p.id_anio_parque = a.id_anio_parque
WHERE p.es_tipo_valido = true
GROUP BY a.anio_referencia
ORDER BY a.anio_referencia
""")

p1_volumen = spark.sql("""
SELECT t.anio_referencia, SUM(c.volumen) AS volumen_total
FROM fact_conteo c
JOIN dim_tiempo t ON c.id_tiempo = t.id_tiempo
GROUP BY t.anio_referencia
ORDER BY t.anio_referencia
""")

p1_ratio = spark.sql("""
SELECT
  parque.anio_referencia,
  parque.total_parque,
  vol.volumen_total,
  ROUND(vol.volumen_total / parque.total_parque, 2) AS ratio_circulacion_parque
FROM (
  SELECT a.anio_referencia, SUM(p.cantidad) AS total_parque
  FROM fact_parque p
  JOIN dim_anio_parque a ON p.id_anio_parque = a.id_anio_parque
  WHERE p.es_tipo_valido = true
  GROUP BY a.anio_referencia
) parque
JOIN (
  SELECT t.anio_referencia, SUM(c.volumen) AS volumen_total
  FROM fact_conteo c
  JOIN dim_tiempo t ON c.id_tiempo = t.id_tiempo
  GROUP BY t.anio_referencia
) vol ON parque.anio_referencia = vol.anio_referencia
ORDER BY parque.anio_referencia
""")

df_p1 = p1_ratio.toPandas()
print(df_p1.to_string(index=False))

growth_parque = (df_p1.loc[df_p1.anio_referencia==2024, 'total_parque'].values[0] /
                 df_p1.loc[df_p1.anio_referencia==2019, 'total_parque'].values[0] - 1) * 100
growth_vol = (df_p1.loc[df_p1.anio_referencia==2024, 'volumen_total'].values[0] /
              df_p1.loc[df_p1.anio_referencia==2019, 'volumen_total'].values[0] - 1) * 100
print(f"\nCrecimiento parque 2019→2024: {growth_parque:.1f}%")
print(f"Crecimiento volumen 2019→2024: {growth_vol:.1f}%")

In [ ]:
# Visualización 1 — Parque vs circulación (normalizado)
df_plot = df_p1.melt(id_vars="anio_referencia",
                     value_vars=["total_parque", "volumen_total"],
                     var_name="metrica", value_name="valor")
df_plot["valor_norm"] = df_plot.groupby("metrica")["valor"].transform(lambda x: x / x.max())

plt.figure(figsize=(8, 5))
sns.barplot(data=df_plot, x="anio_referencia", y="valor_norm", hue="metrica")
plt.title("P1 — Parque vs volumen de circulación (normalizado)")
plt.ylabel("Valor normalizado (0-1)")
plt.xlabel("Año de referencia")
plt.tight_layout()
plt.show()

## P2 — ¿Cómo evolucionó la electrificación del parque?

> ¿Cómo evolucionó la participación de vehículos eléctricos e híbridos en el parque automotor entre 2019 y 2024?

In [ ]:
p2_composicion = spark.sql("""
SELECT
  a.anio_referencia,
  cb.categoria_combustible,
  SUM(p.cantidad) AS cantidad
FROM fact_parque p
JOIN dim_combustible cb ON p.id_combustible = cb.id_combustible
JOIN dim_anio_parque a ON p.id_anio_parque = a.id_anio_parque
WHERE p.es_tipo_valido = true
GROUP BY a.anio_referencia, cb.categoria_combustible
ORDER BY a.anio_referencia, cantidad DESC
""")

p2_electricos_top = spark.sql("""
SELECT p.marca, SUM(p.cantidad) AS cantidad
FROM fact_parque p
JOIN dim_combustible cb ON p.id_combustible = cb.id_combustible
JOIN dim_anio_parque a ON p.id_anio_parque = a.id_anio_parque
WHERE cb.categoria_combustible = 'ELECTRICO'
  AND a.anio_referencia = 2024
  AND p.es_tipo_valido = true
GROUP BY p.marca
ORDER BY cantidad DESC
LIMIT 10
""")

df_p2 = p2_composicion.toPandas()
df_p2_top = p2_electricos_top.toPandas()
print("Composición por combustible:")
print(df_p2.to_string(index=False))
print("\nTop 10 marcas eléctricas 2024:")
print(df_p2_top.to_string(index=False))

In [ ]:
# Visualización 2 — Composición del parque por combustible
plt.figure(figsize=(10, 5))
sns.barplot(data=df_p2, x="anio_referencia", y="cantidad", hue="categoria_combustible")
plt.title("P2 — Composición del parque por categoría de combustible")
plt.ylabel("Cantidad de vehículos")
plt.yscale("log")
plt.tight_layout()
plt.show()

# Detalle eléctricos 2024
plt.figure(figsize=(10, 4))
sns.barplot(data=df_p2_top, x="cantidad", y="marca", hue="marca", legend=False)
plt.title("P2 — Top 10 marcas eléctricas (2024)")
plt.tight_layout()
plt.show()

## P3 — ¿Cómo cambió la velocidad por avenida y franja horaria?

> ¿Cómo cambió la velocidad promedio por avenida y franja horaria entre 2019 y 2024?

In [ ]:
p3_velocidad = spark.sql("""
WITH ubicaciones_comunes AS (
  SELECT id_ubicacion
  FROM (
    SELECT v.id_ubicacion, t.anio_referencia
    FROM fact_velocidad v
    JOIN dim_tiempo t ON v.id_tiempo = t.id_tiempo
    WHERE v.velocidad_promedio > 0
    GROUP BY v.id_ubicacion, t.anio_referencia
  ) x
  GROUP BY id_ubicacion
  HAVING COUNT(DISTINCT anio_referencia) = 2
)
SELECT
  u.dsc_avenida,
  t.anio_referencia,
  t.franja_horaria,
  ROUND(AVG(v.velocidad_promedio), 1) AS vel_prom
FROM fact_velocidad v
JOIN dim_tiempo t ON v.id_tiempo = t.id_tiempo
JOIN dim_ubicacion u ON v.id_ubicacion = u.id_ubicacion
WHERE v.velocidad_promedio > 0
  AND v.id_ubicacion IN (SELECT id_ubicacion FROM ubicaciones_comunes)
GROUP BY u.dsc_avenida, t.anio_referencia, t.franja_horaria
""")

df_p3 = p3_velocidad.toPandas()
print(df_p3.head(10).to_string(index=False))

In [ ]:
# Visualización 3 — Heatmap top 10 avenidas (2024, horas pico)
top_avenidas = (
    df_p3[df_p3.anio_referencia == 2024]
    .groupby("dsc_avenida")["vel_prom"].mean()
    .sort_values(ascending=False)
    .head(10)
    .index.tolist()
)

df_heat = df_p3[
    (df_p3.dsc_avenida.isin(top_avenidas)) &
    (df_p3.franja_horaria.isin(["PICO_MANANA", "PICO_TARDE", "MEDIODIA"]))
].pivot_table(index="dsc_avenida", columns=["anio_referencia", "franja_horaria"], values="vel_prom")

plt.figure(figsize=(12, 6))
sns.heatmap(df_heat, annot=True, fmt=".0f", cmap="RdYlGn")
plt.title("P3 — Velocidad promedio: top 10 avenidas por año y franja")
plt.tight_layout()
plt.show()

## P4 — ¿Mayor flujo implica menor velocidad? *(→ HDFS + SuperSet)*

> ¿Las avenidas con mayor flujo vehicular registran menor velocidad promedio? ¿Cambió esa relación entre 2019 y 2024?

In [ ]:
p4_result = spark.sql("""
SELECT
  u.dsc_avenida,
  t.anio_referencia,
  t.franja_horaria,
  ROUND(AVG(c.volumen_hora), 2) AS vol_hora_prom,
  ROUND(AVG(v.velocidad_promedio), 2) AS vel_prom,
  ROUND(AVG(c.volumen_hora) / NULLIF(AVG(v.velocidad_promedio), 0), 2) AS ratio_congestion
FROM fact_conteo c
JOIN fact_velocidad v
  ON c.id_ubicacion = v.id_ubicacion AND c.id_tiempo = v.id_tiempo
JOIN dim_ubicacion u ON c.id_ubicacion = u.id_ubicacion
JOIN dim_tiempo t ON c.id_tiempo = t.id_tiempo
WHERE v.velocidad_promedio > 0
GROUP BY u.dsc_avenida, t.anio_referencia, t.franja_horaria
""")

p4_result.write.mode("overwrite").parquet(f"{ANALYTICS}/analytics_flujo_velocidad")
print(f"P4 guardada en {ANALYTICS}/analytics_flujo_velocidad")
p4_result.show(5)

## P5 — ¿Dónde se concentra el tránsito y la red de sensores? *(→ HDFS + SuperSet)*

> ¿Qué zonas geográficas concentran el mayor tránsito y cómo se distribuyen los sensores?

In [ ]:
p5_result = spark.sql("""
SELECT
  u.id_ubicacion,
  u.dsc_avenida,
  u.latitud,
  u.longitud,
  t.anio_referencia,
  SUM(c.volumen) AS volumen_total,
  ROUND(AVG(CASE WHEN t.es_hora_pico THEN c.volumen_hora END), 2) AS volumen_hora_pico,
  CASE WHEN COALESCE(s.cnt, 0) > 0 THEN true ELSE false END AS tiene_sensor,
  COALESCE(s.cnt, 0) AS cantidad_sensores
FROM fact_conteo c
JOIN dim_tiempo t ON c.id_tiempo = t.id_tiempo
JOIN dim_ubicacion u ON c.id_ubicacion = u.id_ubicacion
LEFT JOIN (
  SELECT id_ubicacion, anio_referencia, COUNT(*) AS cnt
  FROM dim_sensor
  GROUP BY id_ubicacion, anio_referencia
) s ON c.id_ubicacion = s.id_ubicacion AND t.anio_referencia = s.anio_referencia
GROUP BY u.id_ubicacion, u.dsc_avenida, u.latitud, u.longitud, t.anio_referencia, s.cnt
""")

p5_result.write.mode("overwrite").parquet(f"{ANALYTICS}/analytics_transito_geografico")
print(f"P5 guardada en {ANALYTICS}/analytics_transito_geografico")
p5_result.orderBy("volumen_total", ascending=False).show(5)

## Verificación final

In [ ]:
for nombre in ("analytics_flujo_velocidad", "analytics_transito_geografico"):
    n = spark.read.parquet(f"{ANALYTICS}/{nombre}").count()
    print(f"{nombre}: {n:,} filas")

!hdfs dfs -ls hdfs:///movilidad/analytics